# 실제 도구로 ReAct 에이전트 만들기

이 노트북에서는 문제를 스스로 추론하고 여러 실제 도구 중에서 골라 문제를 해결하는 **ReAct 에이전트**를 만듭니다.

**ReAct**(Reasoning + Acting)는 단순하지만 강력합니다.
1. **추론(Reason)**: LLM이 다음에 수행할 작업을 설명합니다
2. **행동(Act)**: LLM이 도구 호출 요청을 생성하고 Python 코드가 실행합니다
3. **관찰(Observe)**: 도구가 결과를 반환합니다
4. **반복(Repeat)**: 작업이 끝날 때까지 반복합니다

## 사용할 도구

- **MCP Calendar Server**: 일정 관리(앞에서 사용한 9개의 캘린더 도구)
- **RAG 시스템**: 문서에서 정보 검색
- **계산기**: 간단한 산술 연산

에이전트는 사용자 요청에 따라 어떤 도구를 어떤 순서로 사용할지 스스로 결정합니다!

### 실습 목표
앞선 실습의 고정된 단일 도구 실행 흐름을 확장하여 모델이 관찰 결과에 따라 다음 행동을 선택하도록 만듭니다.

`질문 → LLM → Action/Action Input → 도구 실행 → Observation → LLM → ... → Final Answer`

여기서 `Thought`는 모델이 생성한 다음 단계에 대한 설명이며, 실제 내부 추론 전체를 정확히 보여 주는 기록으로 간주하지 않습니다. 종료 조건과 반복 횟수 제한은 노트북의 Python 코드가 관리합니다.


## 설정: 의존성 설치하기

MCP SDK는 캘린더 서버 통신에, `httpx`는 HTTP 통신 관련 기능에 사용됩니다. 설치 셀을 실행한 뒤 import 오류가 발생하면 선택한 Python 커널과 설치 환경이 동일한지 확인하세요.


In [ ]:
!pip3 install -q mcp httpx

Llama Stack 엔드포인트와 Calendar MCP Server 엔드포인트를 정의합니다.

Llama Stack은 모델 추론과 RAG API를 제공하고, MCP Calendar Server는 일정 조회 및 변경 도구를 제공합니다. 두 서비스의 역할과 주소가 서로 다릅니다. 연결 완료 출력은 클라이언트 설정을 나타내며 실제 API 성공 여부는 이후 도구 실행으로 검증합니다.


In [ ]:
# 모델 추론/RAG API와 MCP 서버는 별개 서비스입니다.
from llama_stack_client import LlamaStackClient
from llama_stack_client.lib.inference.event_logger import EventLogger
from mcp import ClientSession
from mcp.client.sse import sse_client
import json
from typing import Dict, Any, List
import re
import logging
from datetime import datetime

# HTTP 로그 출력 줄이기
logging.getLogger("httpx").setLevel(logging.WARNING)

# Llama Stack 서버에 연결
base_url = "http://llama-stack-service:8321"
client = LlamaStackClient(base_url=base_url)
model = "vllm-llama32/llama32"

# MCP 서버 URL
mcp_server_url = "http://canopy-mcp-calendar-mcp-server:8080/sse"

print(f"✅ Llama Stack 연결 완료: {base_url}")
print(f"✅ 사용 모델: {model}")
print(f"✅ MCP 서버: {mcp_server_url}")

## 도구 설정하기

세 가지 도구 범주(계산기, MCP 캘린더, RAG)에 대한 래퍼 함수를 만들어 보겠습니다.

래퍼 함수는 서로 다른 도구의 반환값을 공통 문자열 형식으로 맞춥니다. 이렇게 하면 ReAct 루프는 도구 내부 구현을 알 필요 없이 결과를 `Observation`으로 대화 이력에 추가할 수 있습니다.


### 도구 1: 계산기

`calculator_tool()`은 모델이 전달한 연산 종류와 두 숫자를 받아 사칙연산을 수행합니다. 정상 결과는 `Result: ...`, 실패는 `Error: ...` 형태로 반환하므로 모델은 실제 관찰값을 바탕으로 답변할 수 있습니다.


In [ ]:
# 정상값과 오류를 모두 문자열로 반환하여 관찰 결과로 전달할 수 있게 합니다.
def calculator_tool(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic operations."""
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return "Error: Cannot divide by zero"
            result = a / b
        else:
            return f"Error: Unknown operation '{operation}'"
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ 계산기 도구 준비 완료")

### 도구 2: MCP 캘린더 도구

`get_mcp_tools_description()`으로 도구 목록을 조회하고 이름 앞에 `mcp_` 접두사를 붙여 다른 도구와 구분합니다. 실행 시에는 `execute_tool()`이 접두사를 제거하여 서버의 실제 도구 이름으로 호출합니다.

현재 구현은 이름과 설명을 프롬프트에 전달합니다. 입력 스키마 전체를 별도로 전달하지 않으므로 모델이 잘못된 카테고리나 인자를 생성할 가능성이 있습니다. 이는 뒤의 어려운 요청 실습에서 관찰할 수 있는 한계입니다.


In [ ]:
# 서버에서 받은 도구 설명에 mcp_ 접두사를 추가하고, 실제 실행 시 별도 세션을 사용합니다.
async def call_mcp_tool(tool_name: str, arguments: Dict[str, Any]) -> str:
    """Call an MCP calendar tool and return the result as a string."""
    try:
        async with sse_client(mcp_server_url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments=arguments)
                return "\n".join([content.text for content in result.content])
    except Exception as e:
        return f"Error calling MCP tool: {str(e)}"

async def get_mcp_tools_description() -> str:
    """Get descriptions of all available MCP tools."""
    async with sse_client(mcp_server_url) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()

            descriptions = []
            for tool in tools.tools:
                descriptions.append(f"  - mcp_{tool.name}: {tool.description}")

            return "\n".join(descriptions)

# 사용 가능한 MCP 도구를 가져와 표시
mcp_tools_desc = await get_mcp_tools_description()
print("✅ 사용 가능한 MCP 일정 관리 도구:")
print(mcp_tools_desc)

### 도구 3: RAG(문서 검색)

에이전트가 문서를 검색할 수 있도록 RAG 시스템을 설정하겠습니다.
**RAG(Retrieval-Augmented Generation)**는 모델에 질문할 때 외부 문서에서 관련 정보를 검색하여 답변의 근거를 제공하는 방식입니다. 


In [ ]:
import io
import time

# 실습 전용 설정
store_name = "canopy_agentic_rag"
embedding_model_id = "sentence-transformers/nomic-ai/nomic-embed-text-v1.5"
embedding_dimension = 768
TEST_QUERY = "What is the structure of Canopy in botany?"
vector_store = None
rag_ready = False


def create_lab_store():
    store = client.vector_stores.create(
        name=store_name,
        extra_body={
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
            "provider_id": "milvus",
            "vector_db_id": store_name
        }
    )
    if store.id != store_name:
        raise RuntimeError(f"예상과 다른 저장소 ID: {store.id}")
    return store


def test_search(store):
    return client.vector_stores.search(
        vector_store_id=store.id,
        query=TEST_QUERY,
        max_num_results=5,
        search_mode="vector"
    )


# 1. 실습용 저장소 조회 또는 생성
selected_store = next(
    (vs for vs in client.vector_stores.list().data if vs.id == store_name),
    None
)
if selected_store is None:
    selected_store = create_lab_store()
    print("✅ 실습용 벡터 저장소 생성 완료")

# 2. 등록 문서의 상태 및 실제 검색 가능 여부 확인
existing_files = client.vector_stores.files.list(vector_store_id=selected_store.id).data
has_completed = any(f.status == "completed" for f in existing_files)
has_in_progress = any(f.status == "in_progress" for f in existing_files)
search_results = test_search(selected_store) if has_completed else None

if has_completed and not search_results.data:
    # 검색 API 예외는 그대로 전파하며, 결과가 비어 있을 때만 재생성합니다.
    if has_in_progress or selected_store.id != store_name or selected_store.name != store_name:
        raise RuntimeError("인덱싱 진행 중이거나 실습용 저장소가 아니므로 삭제할 수 없습니다.")
    print("⚠️ 검색 결과 없음: 실습 전용 저장소 재생성")
    client.vector_stores.delete(vector_store_id=selected_store.id)
    selected_store = create_lab_store()
    has_completed = False

# 3. 정상적으로 검색 가능한 문서가 없으면 샘플 문서를 등록
if not has_completed:
    if has_in_progress:
        raise RuntimeError("인덱싱 진행 중입니다. 완료 후 다시 실행하세요.")

    document = """
Canopy in Botany

In botany, a canopy is the aboveground portion of a plant
community formed by the crowns of trees and other vegetation.

Structure of a Forest Canopy

The vertical structure of a tropical rainforest typically
consists of four layers:

1. Emergent Layer:
   The tallest trees extending above the main forest canopy.

2. Canopy Layer:
   A dense layer of overlapping tree crowns that intercepts
   most of the sunlight.

3. Understory Layer:
   Smaller trees, shrubs, and plants adapted to lower
   light conditions.

4. Forest Floor:
   The lowest layer, containing decomposing organic matter
   and shade-tolerant vegetation.

Functions of the Canopy

The canopy regulates temperature, humidity, sunlight,
and rainfall interception. It provides habitats for
numerous species and plays an important role in
photosynthesis and carbon storage.
"""
    uploaded_file = client.files.create(
        file=("canopy_botany.txt", io.BytesIO(document.encode("utf-8"))),
        purpose="assistants"
    )
    indexed_file = client.vector_stores.files.create(
        vector_store_id=selected_store.id,
        file_id=uploaded_file.id,
        chunking_strategy={
            "type": "static",
            "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 64}
        }
    )

    # 최대 120초 동안 인덱싱 완료 대기 (API 호출 시간 제외)
    for _ in range(60):
        if indexed_file.status != "in_progress":
            break
        time.sleep(2)
        indexed_file = client.vector_stores.files.retrieve(
            vector_store_id=selected_store.id, file_id=uploaded_file.id
        )
    if indexed_file.status == "in_progress":
        raise TimeoutError("문서 인덱싱 완료 대기 시간 초과")
    if indexed_file.status != "completed":
        raise RuntimeError(f"문서 인덱싱 실패: {indexed_file}")

    print("✅ 문서 임베딩 및 인덱싱 완료")
    search_results = test_search(selected_store)
else:
    print("✅ 기존 문서 재사용")

# 4. 검색 성공 후에만 RAG 활성화
if not search_results.data:
    raise RuntimeError("벡터 저장소에서 관련 문서를 검색하지 못했습니다.")

vector_store = selected_store
rag_ready = True
print(f"✅ RAG 시스템 초기화 완료: {vector_store.id}")
print(f"검색 결과: {len(search_results.data)}개")
print(search_results.data)


RAG 문서 검색 도구를 아래와 같이 설정합니다.

`rag_search_tool()`은 초기화된 Vector Store에 질의를 보내 최대 `max_results`개의 검색 결과를 반환합니다. 검색 결과는 모델이 읽을 관찰 정보이며 이 함수 자체가 최종 답변을 생성하지는 않습니다. 벡터 검색은 상대적으로 가까운 문서를 반환하므로 결과가 있다고 해서 질문과 반드시 관련된 것은 아닙니다.


In [ ]:
# 검색 결과를 문자열로 구성해 반환하며 최종 응답 생성은 ReAct 루프에서 수행합니다.
def rag_search_tool(query: str, max_results: int = 3) -> str:
    """Search through documents using RAG and return relevant information."""
    if not vector_store:
        return "Error: RAG system not initialized. Please set up a vector store first."
    
    try:
        # 벡터 데이터베이스 검색
        search_results = client.vector_stores.search(
            vector_store_id=vector_store.id,
            query=query,
            max_num_results=max_results,
            search_mode="vector"
        )
        
        if not search_results.data:
            return "No relevant documents found."
        
        # 결과 정리
        results = []
        for i, result in enumerate(search_results.data, 1):
            content = result.content if hasattr(result, 'content') else str(result)
            results.append(f"Result {i}: {content}")
        
        return "\n\n".join(results)
    except Exception as e:
        return f"Error performing RAG search: {str(e)}"

print("✅ RAG 검색 도구 준비 완료")

## ReAct 에이전트 만들기

이제 사용할 도구를 스스로 선택하는 ReAct 에이전트를 만들어 보겠습니다!

System Prompt에는 사용 가능한 도구, 사용 조건, 응답 형식 및 현재 날짜가 포함됩니다. `mcp_tools_desc`가 동적으로 삽입되는 부분을 확인하세요.

이 실습은 일반 텍스트 응답을 직접 파싱하는 방식입니다. `Action`에는 도구명, `Action Input`에는 JSON 인자를, `Final Answer`에는 최종 응답을 넣도록 요청합니다. 프롬프트는 행동을 유도할 뿐 형식 준수나 도구 실행 성공을 강제하지는 않습니다.


In [ ]:
# 도구 설명과 ReAct 응답 규칙은 영어로 유지합니다. 

def create_react_system_prompt() -> str:
    """Create a system prompt for the ReAct agent."""

    return f"""You are a helpful assistant that can use tools
to help students with their calendar and studies.

Today is {datetime.today().strftime('%Y-%m-%d')}.

## Available Tools

1. calculator
   - Performs arithmetic operations.
   - Input: {{"operation": "add"|"subtract"|"multiply"|"divide",
     "a": number, "b": number}}

2. MCP Calendar Tools:
{mcp_tools_desc}

3. rag_search
   - Searches knowledge base documents.
   - Input: {{"query": "search query", "max_results": 3}}

## Tool Usage Rules

- For knowledge questions, use rag_search.
- For calendar requests, use MCP calendar tools.
- For calculations, use calculator.
- Use only tools relevant to the user's request.
- Follow each tool's input schema.
- Do not apply unnecessary calendar filters.
- Do not modify calendar events unless explicitly requested.
- Complete all parts of the user's request.
- Once you have enough information, provide a Final Answer.
- Do not repeat successful tool calls unnecessarily.
- Never fabricate tool results or claim success without confirmation.

## Response Format

When a tool is needed, use exactly:

Thought: [Brief description of the next step]
Action: [tool_name]
Action Input: {{"key": "value"}}

Use valid JSON. Use {{}} if no arguments are needed.
Stop after Action Input and wait for the actual Observation.

When sufficient information is available, respond with:

Final Answer: [answer based on actual tool results]

Never combine a tool call and Final Answer in one response.
If a tool fails, report the error or retry with valid arguments.
"""

print("✅ ReAct 시스템 프롬프트 생성 완료")


## ReAct 루프 구현

세 함수가 직접 구현한 에이전트의 핵심입니다.

| 함수 | 역할 |
|---|---|
| `parse_react_response()` | 정규식으로 Thought, Action, JSON 입력, Final Answer 추출 |
| `execute_tool()` | 도구 이름에 따라 계산기, MCP, RAG 호출 |
| `run_react_agent()` | 모델 호출, 관찰 결과 추가, 반복 및 종료 관리 |

기본 최대 반복 횟수는 5회입니다. 한 반복은 한 번의 모델 호출이며, 최종 답변을 만드는 호출도 횟수에 포함됩니다. 화면의 관찰 결과는 최대 300자로 줄여 표시하지만 대화 이력에는 전체 결과를 넣습니다.

In [ ]:
# 파싱, 도구 라우팅, 반복 실행을 직접 구현합니다. 각 반복에서 실제 관찰 결과를 대화에 추가합니다.
def parse_react_response(response: str) -> Dict[str, Any]:
    """Parse the LLM's ReAct-formatted response."""
    result = {}
    
    # Thought 추출
    thought_match = re.search(r'Thought:\s*(.+?)(?=\n(?:Action|Final Answer):|$)', response, re.DOTALL)
    if thought_match:
        result['thought'] = thought_match.group(1).strip()
    
    # Final Answer가 있는지 확인
    final_answer_match = re.search(r'Final Answer:\s*(.+)', response, re.DOTALL)
    if final_answer_match:
        result['final_answer'] = final_answer_match.group(1).strip()
        return result
    
    # Action 추출
    action_match = re.search(r'Action:\s*(.+?)(?=\n|$)', response)
    if action_match:
        result['action'] = action_match.group(1).strip()
    
    # Action Input 추출
    action_input_match = re.search(r'Action Input:\s*({.+})', response, re.DOTALL)
    if action_input_match:
        try:
            json_str = action_input_match.group(1).strip()
            # 흔한 JSON 형식 문제 정리
            json_str = re.sub(r'\n\s*', ' ', json_str)  # 줄바꿈 제거
            result['action_input'] = json.loads(json_str)
        except json.JSONDecodeError as e:
            result['error'] = f"Failed to parse action input: {e}"
    
    return result


async def execute_tool(action: str, action_input: Dict[str, Any]) -> str:
    """Execute the appropriate tool based on the action."""
    
    if action == "calculator":
        return calculator_tool(
            action_input.get("operation"),
            action_input.get("a"),
            action_input.get("b")
        )
    
    elif action.startswith("mcp_"):
        # MCP 캘린더 도구
        tool_name = action[4:]  # 'mcp_' 접두사 제거
        arguments = action_input.get("arguments", action_input)
        return await call_mcp_tool(tool_name, arguments)
    
    elif action == "rag_search":
        return rag_search_tool(
            action_input.get("query"),
            action_input.get("max_results", 3)
        )
    
    else:
        return f"Error: Unknown tool '{action}'"


async def run_react_agent(user_question: str, max_iterations: int = 5, verbose: bool = True):
    """Run the ReAct agent loop."""
    
    if verbose:
        print("="*70)
        print(f"사용자 질문: {user_question}")
        print("="*70)
    
    system_prompt = create_react_system_prompt()
    conversation_history = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]
    
    for iteration in range(max_iterations):
        if verbose:
            print(f"\n{'='*70}")
            print(f"반복 실행 {iteration + 1}")
            print("="*70)
        
        # LLM 응답 받기
        response = client.chat.completions.create(
            model=model,
            messages=conversation_history,
            stream=False,
        )
        
        llm_output = response.choices[0].message.content
        
        if verbose:
            print(f"\n{llm_output}")
        
        # 응답 파싱
        parsed = parse_react_response(llm_output)
        
        # 최종 답변이 있는지 확인
        if 'final_answer' in parsed:
            if verbose:
                print("\n" + "="*70)
                print("최종 답변:")
                print("="*70)
                print(parsed['final_answer'])
                print("="*70)
            return parsed['final_answer']
        
        # action이 있으면 실행
        if 'action' in parsed and 'action_input' in parsed:
            action = parsed['action']
            action_input = parsed['action_input']
            
            if verbose:
                print(f"\n🔧 실행 중: {action}")
                print(f"📥 입력: {json.dumps(action_input, indent=2)}")
            
            # 도구 실행
            tool_result = await execute_tool(action, action_input)
            
            if verbose:
                print(f"\n📊 관찰 결과:")
                result_preview = tool_result[:300] + "..." if len(tool_result) > 300 else tool_result
                print(result_preview)
            
            # 관찰 결과를 대화에 추가
            observation_text = f"Observation: {tool_result}"
            conversation_history.append({"role": "assistant", "content": llm_output})
            conversation_history.append({"role": "user", "content": observation_text})
        else:
            # LLM이 형식을 제대로 따르지 않음
            error_msg = "Please follow the Thought/Action/Action Input format or provide a Final Answer."
            if verbose:
                print(f"\n⚠️  Thought/Action/Action Input 형식을 따르거나 Final Answer를 제공하세요.")
            conversation_history.append({"role": "assistant", "content": llm_output})
            conversation_history.append({"role": "user", "content": error_msg})
    
    if verbose:
        print("\n" + "="*70)
        print("⚠️  경고: 최종 답변 없이 최대 반복 횟수에 도달했습니다")
        print("="*70)
    
    return "I couldn't find an answer within the iteration limit."

print("✅ ReAct 에이전트 준비 완료!")

## 에이전트 테스트하기!

이제 에이전트가 실제로 동작하는 모습을 보겠습니다. 각 작업에 맞는 도구를 스스로 고르는 과정을 지켜보세요.

각 예제는 별도의 대화 이력으로 시작합니다. 사용자 질문에 맞는 `Action`이 선택되었는지, `Action Input`이 유효한지, 실제 `Observation`이 돌아왔는지 확인한 뒤 최종 답변을 검토하세요.


### 예제 1: 계산기 사용하기

에이전트는 이 작업에 계산기가 필요하다는 것을 알아차려야 합니다.

기대 흐름은 `calculator` 호출 후 최종 답변입니다. 847 × 923 = 781,781이 실제 도구 결과로 반환되는지 확인하세요.


In [ ]:
# 단일 계산 도구의 선택과 실제 연산 결과를 확인합니다.
await run_react_agent("What is 847 multiplied by 923?")

### 예제 2: MCP 캘린더 사용하기

에이전트는 캘린더 MCP 도구를 사용해야 합니다.

앞으로 7일간의 일정을 묻는 요청입니다. 기대하는 도구는 `mcp_get_upcoming_events`이며 `days`에 7을 전달할 수 있습니다. 결과 건수는 실행 시점과 실제 캘린더 데이터에 따라 달라집니다.


In [ ]:
# 일정 조회 도구의 입력과 반환 결과를 확인합니다.
await run_react_agent("What events do I have coming up in the next 7 days?")

### 예제 3: RAG 검색 사용하기

에이전트는 RAG로 문서를 검색할 수 있으며, 이를 Agentic RAG라고도 부릅니다.
앞서 도구 3에서 식물학의 Canopy에 관한 실습용 문서를 벡터 저장소에 등록하고, 임베딩 및 인덱싱을 수행했습니다.

에이전트는 RAG 검색 도구를 호출하여 벡터 저장소에서 관련 문서를 검색하고, 검색된 정보를 기반으로 답변을 생성합니다.

기대 흐름은 `rag_search` 호출 → Canopy 문서 검색 → 최종 답변입니다. 검색 결과에는 열대우림의 수직 구조(돌출층, 캐노피층, 하층 식생, 숲 바닥)가 포함되어 있습니다. 충분한 문서를 얻은 뒤 관련 없는 캘린더 호출을 반복한다면 모델의 도구 선택 또는 종료 판단 문제를 의심해 볼 수 있습니다.


In [ ]:
# 저장소의 검색 결과를 활용해 답변하는지 확인합니다.
await run_react_agent("What is the structure of Canopy in botany?")

### 예제 4: 여러 단계의 추론

이 작업을 하려면 에이전트가 다음을 수행해야 합니다.
1. 캘린더에서 일정 검색
2. 계산 수행

캘린더가 반환한 일정 개수를 계산기의 입력으로 사용해야 합니다. 첫 번째 결과가 두 번째 도구 호출에 영향을 준다는 점이 단일 도구 실습과 다른 핵심입니다. 실제 일정 수가 N개라면 최종 답은 2×N입니다.


In [ ]:
# 일정 조회 결과를 다음 계산 도구의 입력으로 활용하는 예제입니다.
await run_react_agent("How many events do I have in the next week, multiplied by 2?")

### 예제 5: 어려운 요청

에이전트에게 조금 더 어려운 요청 몇 가지를 해 봅니다.  
실행해 보고 에이전트가 해내는지 확인하세요. 요청은 실행할 때마다 성공할 수도, 실패할 수도 있습니다!
해내지 못한다면 왜 실패했다고 생각하나요?

첫 번째 요청은 일정 조회와 Canopy 지식 검색을 한 답변에 결합하는 복합 작업입니다. 두 번째는 실제 캘린더에 일정을 생성하는 쓰기 작업으로, 종료 시간이 명시되지 않아 모델이 추가 질문을 하거나 입력을 추론할 수 있습니다.

성공했다고 답변해도 실제 도구 실행 로그와 캘린더를 확인하세요. 반복 실행하면 중복 일정이 만들어질 수 있습니다. 최대 반복 횟수에 도달했다면 무조건 반복 횟수를 늘리기보다 도구 선택, 입력 JSON, 관찰 결과, 종료 판단 중 어느 단계가 실패했는지 살펴보세요.


In [ ]:
# 두 가지 작업을 모두 완료했는지 각각의 도구 로그를 확인합니다.
await run_react_agent("Tell me about my schedule and what Canopy means")

In [ ]:
# 일정 생성은 실제 쓰기 작업이며 반복 실행 시 중복 일정이 생길 수 있습니다.
await run_react_agent("Create a meeting for tomorrow at 2pm called 'Team Sync'")

## 실제 환경에서의 시사점

에이전트 방식은 환상적으로 보이고 실제로도 그렇지만, 에이전트 워크플로를 사용할 때 알아 두어야 할 단점도 있습니다.

- **비용**: 반복 1회 = LLM 호출 1회 = $$$
- **지연 시간**: 추론이 많을수록 응답이 느려집니다
- **신뢰성**: 컨텍스트가 많아지므로 에이전트가 실수하기 쉬워지며, 특히 도구가 많을 때 그렇습니다
- **디버깅**: 여러 단계의 추론은 디버깅하기 어렵습니다

### 장애 분석 체크포인트
| 증상 | 확인할 항목 |
|---|---|
| `Connection error` | Llama Stack/MCP Service 주소와 배포 상태 |
| 인덱싱 `failed` | 임베딩 모델, 차원, Llama Stack 및 Milvus 로그 |
| 도구 인자 검증 오류 | 실제 MCP 도구 설명과 입력 스키마 |
| 반복 횟수 초과 | 잘못된 응답 형식, 중복 호출, 관찰 결과, 최종 답변 생성 |
| 예약 완료 응답과 실제 데이터 불일치 | `create_event` 호출 및 실제 캘린더 데이터 |

다음 실습은 응답 파싱, 도구 라우팅, 실행 루프 등의 구현을 LangGraph와 Llama Stack에 위임하는 방법을 보여 줍니다.


## 직접 해 보세요!

이제 여러분 차례입니다. 다음과 같은 질문을 시도해 보세요.
- 여러 다른 도구가 필요한 질문
- 어떤 도구를 사용할지 추론해야 하는 질문
- 캘린더, 지식, 계산을 결합해야 하는 질문

에이전트가 얼마나 잘 처리하는지 확인해 보세요!

**복습:** 한 번의 LLM 호출만으로 복합 요청을 완료하기 어려운 이유를 설명해 보세요. 도구를 실행한 실제 결과와 모델이 생성한 설명을 어떻게 구분할 수 있을까요? 일정 변경 요청은 실습용 데이터에만 수행하세요.


In [ ]:
# YOUR QUESTION HERE를 직접 만든 자연어 질문으로 바꿔 테스트합니다.
# 여러분 차례입니다! 직접 질문해 보세요:
await run_react_agent("YOUR QUESTION HERE")